In [24]:
import duckdb

con = duckdb.connect()

In [25]:
ref_file = "all_hits_drugs.csv"
filtered_file = "citation_pairs.csv"
unfiltered_file = "drug_docs_with_refs.csv"

con.execute(f"""
    CREATE OR REPLACE TABLE ref AS
    SELECT
        row_number() OVER () AS row_id,
        *
    FROM read_csv_auto('{ref_file}', header=True);
    
    CREATE OR REPLACE TABLE filtered_pairs AS
    SELECT *
    FROM read_csv_auto('{filtered_file}', header=True);
    
    CREATE OR REPLACE TABLE unfiltered_docs AS
    SELECT *
    FROM read_csv_auto('{unfiltered_file}', header=True);
""")

In [26]:
con.execute("""
ALTER TABLE ref
ADD COLUMN DOI_present_regex BOOLEAN;
""")

In [27]:
con.execute("""
    CREATE OR REPLACE TABLE filtered_dois AS
    SELECT DISTINCT hit_paper AS doi FROM filtered_pairs
    UNION
    SELECT DISTINCT drug_paper AS doi FROM filtered_pairs;
""")

In [28]:
con.execute("""
    CREATE OR REPLACE TABLE unfiltered_dois AS
    SELECT DISTINCT doi
    FROM unfiltered_docs
    WHERE doi IS NOT NULL AND trim(doi) <> '';
""")


In [29]:
con.execute("""
    CREATE OR REPLACE TABLE ref_dois_expanded AS
    SELECT
        r.row_id,
        trim(unnest(str_split(r.ref_dois, ';'))) AS doi
    FROM ref r;
""")


In [30]:
# Load patterns from hit_synonyms.txt
with open('hit_synonyms.txt', 'r') as f:
    patterns = [line.strip() for line in f if line.strip()]

# Create pattern names by removing \b boundaries
pattern_names = [pattern.replace('\\b', '') for pattern in patterns]

print(f"Loaded {len(patterns)} patterns.")
# Create regex pattern
combined_pattern = '|'.join(patterns)
print(combined_pattern)

Loaded 11 patterns.
\bhigh[\s-]?throughput\b|\bHTS\b|\bscreen(ing|ed|s)?\b|\bhit(s)?\b|\bidentification\b|\boptimi(s|z)ation\b|\bseries\b|\bcompound\b|\bcandidates?\b|\bfragment\b|\bphenotyp(e|ic)\b


In [31]:
con.execute(f"""
    CREATE OR REPLACE TABLE combined_doi_matches AS
    WITH doi_title_matches AS (
        SELECT 
            re.row_id,
            COUNT(CASE WHEN regexp_matches(ud.title, '{combined_pattern}', 'i') THEN 1 END) as regex_match_count
        FROM ref_dois_expanded re
        LEFT JOIN unfiltered_docs ud ON ud.doi = re.doi
        GROUP BY re.row_id
    )
    SELECT
        r.row_id,
        CASE 
            WHEN r.ref_dois IS NULL OR trim(r.ref_dois) = '' THEN NULL
            WHEN (SELECT COUNT(fd.doi) FROM filtered_dois fd 
                  WHERE fd.doi IN (SELECT doi FROM ref_dois_expanded WHERE row_id = r.row_id)) > 0 THEN TRUE
            ELSE FALSE
        END AS DOI_present_filtered,

        CASE 
            WHEN r.ref_dois IS NULL OR trim(r.ref_dois) = '' THEN NULL
            WHEN dtm.regex_match_count > 0 THEN TRUE
            ELSE FALSE
        END AS DOI_present_regex,

        CASE 
            WHEN r.ref_dois IS NULL OR trim(r.ref_dois) = '' THEN NULL
            WHEN (SELECT COUNT(ud.doi) FROM unfiltered_dois ud 
                  WHERE ud.doi IN (SELECT doi FROM ref_dois_expanded WHERE row_id = r.row_id)) > 0 THEN TRUE
            ELSE FALSE
        END AS DOI_present_unfiltered

    FROM ref r
    LEFT JOIN doi_title_matches dtm ON dtm.row_id = r.row_id
    ORDER BY r.row_id;
""")

In [33]:
result = con.execute("""
    SELECT
        r.*,
        m.DOI_present_filtered,
        m.DOI_present_regex,
        m.DOI_present_unfiltered
    FROM ref r
    LEFT JOIN combined_doi_matches m USING (row_id)
    ORDER BY r.row_id;
""").df()

result

,row_id,paper,clinical_candidate,candidate_smiles,hit,hit_smiles,target,candidate_chembl_id,disease_area,entry,hit_chembl_id,method,ref,ref_dois,root_name,DOI_present_regex,DOI_present_filtered,DOI_present_regex_1,DOI_present_unfiltered
0,1,Brown_2018,ABBV-075/mivebresib,CCS(=O)(=O)Nc1ccc(Oc2ccc(F)cc2F)c(-c2cn(C)c(=O...,CHEMBL4068525,CNc1cc(=O)n(C)nc1-c1ccccc1,BRD4,CHEMBL3987016,oncology,1,CHEMBL4068525,fragment_screen,"13,14",10.1021/acs.jmedchem.7b00017;10.1021/acs.jmedc...,None,<NA>,False,False,True
1,2,Brown_2018,AZD5153,COc1nnc2ccc(N3CCC(c4ccc(OCCN5CCN(C)C(=O)[C@H]5...,AZD3514,CC(=O)N1CCN(CCOc2ccc(C3CCN(C4=Nn5c(nnc5C(F)(F)...,BRD4,CHEMBL4078100,oncology,2,CHEMBL2346976,known,15,10.1021/acs.jmedchem.6b00070,None,<NA>,True,True,True
2,3,Brown_2018,CPI-0610,Cc1noc2c1-c1ccccc1C(c1ccc(Cl)cc1)=N[C@H]2CC(N)=O,CHEMBL2431088,Cc1noc(N)c1-c1ccccc1,BRD4,CHEMBL4303404,oncology,3,CHEMBL2431088,fragment_screen,16,10.1021/acs.jmedchem.5b01882,None,<NA>,True,True,True
3,4,Brown_2018,EOS200271/PF-06840003,O=C1CC(c2c[nH]c3ccc(F)cc23)C(=O)N1,CHEMBL1374439,O=C1CC(c2c[nH]c3ccccc23)C(=O)N1,IDO-1 inhibitor,CHEMBL4086143,oncology,4,CHEMBL1374439,random_screen,17,10.1021/acs.jmedchem.7b00974,None,<NA>,True,True,True
4,5,Brown_2018,PQR309,Nc1cc(C(F)(F)F)c(-c2nc(N3CCOCC3)nc(N3CCOCC3)n2...,CHEMBL1910991,FC(F)(F)c1nc2ccccc2n1-c1nc(N2CCOCC2)nc(N2CCOCC...,PI3K/mTOR,CHEMBL4084907,oncology,5,CHEMBL1910991,known,18,10.1021/acs.jmedchem.7b00930,None,<NA>,True,True,True
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
217,218,Opera_2001,CAPTOPRIL,C[C@H](CS)C(=O)N1CCC[C@H]1C(=O)O,SQ13297,C[C@H](CC(=O)O)C(=O)N1CCC[C@H]1C(=O)O,Angiotensin Converting Enzyme,None,None,<NA>,None,None,None,None,None,<NA>,<NA>,<NA>,<NA>
218,219,Opera_2001,HALOPERIDOL,O=C(CCCN1CCC(O)(c2ccc(Cl)cc2)CC1)c1ccc(F)cc1,R1187,CCOC(=O)C1(c2ccccc2)CCN(CCCC(=O)c2ccccc2)CC1,neuroleptic,None,None,<NA>,None,None,None,None,None,<NA>,<NA>,<NA>,<NA>
219,220,Opera_2001,BENPERIDOL,O=C(CCCN1CCC(n2c(=O)[nH]c3ccccc32)CC1)c1ccc(F)cc1,HALOPERIDOL,O=C(CCCN1CCC(O)(c2ccc(Cl)cc2)CC1)c1ccc(F)cc1,neuroleptic,None,None,<NA>,None,None,None,None,None,<NA>,<NA>,<NA>,<NA>
220,221,Opera_2001,PIMOZIDE,O=c1[nH]c2ccccc2n1C1CCN(CCCC(c2ccc(F)cc2)c2ccc...,BENPERIDOL,O=C(CCCN1CCC(n2c(=O)[nH]c3ccccc32)CC1)c1ccc(F)cc1,neuroleptic,None,None,<NA>,None,None,None,None,None,<NA>,<NA>,<NA>,<NA>


In [34]:
# Export all_hits_drugs with DOI flags appended
con.execute("""
    COPY result
    TO 'all_hits_drugs_with_flags.csv'
    (HEADER, DELIMITER ',');
""")

print("Saved all_hits_drugs_with_flags.csv")


Saved all_hits_drugs_with_flags.csv
